# GitHub Flow와 PR 리뷰

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.


## 준비

로컬 bare 저장소를 원격 저장소처럼 사용합니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# bare 저장소는 원격 역할만 담당합니다.
remote = root / "team.git"
git("init", "--bare", "--initial-branch=main", remote.as_posix())
author = root / "author"
git("clone", remote.as_posix(), author.as_posix())
configure(author)
(author / "README.md").write_text("# 주문 서비스\n", encoding="utf-8")
template = author / ".github" / "pull_request_template.md"
template.parent.mkdir()
template.write_text("## 변경\n- 변경 이유\n\n## 확인\n- [ ] 로컬 실행\n", encoding="utf-8")
git("add", "README.md", template.relative_to(author).as_posix(), cwd=author)
git("commit", "-m", "docs: 저장소와 PR 템플릿 시작", cwd=author)
git("push", "-u", "origin", "main", cwd=author)
print(git("remote", "-v", cwd=author).splitlines()[0])

origin	/var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/git-lesson-icolm0jj/team.git (fetch)


## 기본 실행

기능 브랜치를 만들고 커밋한 뒤 원격에 게시합니다.

In [3]:
# 타입힌트는 실행 시 값을 검사하지 않으므로 별도 입력 검증이 필요합니다.
git("switch", "-c", "feature/order-summary", cwd=author)
(author / "summary.py").write_text(
    """def summarize(count: int) -> str:
    return f'주문 {count}건'
""", encoding="utf-8"
)
git("add", "summary.py", cwd=author)
git("commit", "-m", "feat: 주문 요약 함수 추가", cwd=author)
git("push", "-u", "origin", "feature/order-summary", cwd=author)
print(git("branch", "-vv", cwd=author))

* feature/order-summary 84ce428 [origin/feature/order-summary] feat: 주문 요약 함수 추가
  main                  8eac6ee [origin/main] docs: 저장소와 PR 템플릿 시작


## PR 자료와 변경 범위

기본 브랜치에 준비된 템플릿을 읽고 기능 코드 차이만 확인합니다.

In [4]:
# 기본 브랜치의 템플릿을 채우고 세 점 비교로 기능 브랜치의 변경을 확인합니다.
pr_body = template.read_text(encoding="utf-8").replace(
    "- 변경 이유", "- 주문 건수 요약 함수 추가"
).replace("- [ ] 로컬 실행", "- [x] 로컬 실행")
print(git("diff", "--stat", "origin/main...HEAD", cwd=author))
print(pr_body)

summary.py | 2 ++
 1 file changed, 2 insertions(+)
## 변경
- 주문 건수 요약 함수 추가

## 확인
- [x] 로컬 실행



## 실패 재현

리뷰어가 빈 문자열 입력에 대한 설명과 처리가 없음을 실제 diff에서 발견합니다.

In [5]:
# 리뷰어의 복제본에서 빈 문자열을 직접 전달해 잘못된 출력을 재현합니다.
reviewer = root / "reviewer"
git("clone", remote.as_posix(), reviewer.as_posix())
configure(reviewer)
git("switch", "feature/order-summary", cwd=reviewer)
before = (reviewer / "summary.py").read_text(encoding="utf-8")
print(before)
print("리뷰 의견: count가 빈 문자열이면 주문 건수가 비어 보입니다.")
print("재현:", subprocess.check_output(
    [sys.executable, "-c", "from summary import summarize; print(summarize(''))"],
    cwd=reviewer, text=True,
).strip())

def summarize(count: int) -> str:
    return f'주문 {count}건'

리뷰 의견: count가 빈 문자열이면 주문 건수가 비어 보입니다.
재현: 주문 건


## 수정

작성자가 입력을 정수로 제한하고 리뷰어가 원격 변경을 받아 병합합니다.

In [6]:
# 수정 커밋을 공유하고 실제 TypeError와 원격 main 갱신을 함께 확인합니다.
(author / "summary.py").write_text(
    """def summarize(count: int) -> str:
    if not isinstance(count, int):
        raise TypeError('count는 정수여야 합니다')
    return f'주문 {count}건'
""", encoding="utf-8"
)
git("add", "summary.py", cwd=author)
git("commit", "-m", "fix: 주문 건수 입력 확인", cwd=author)
git("push", cwd=author)
git("fetch", "origin", cwd=reviewer)
git("switch", "main", cwd=reviewer)
git("merge", "--no-ff", "origin/feature/order-summary", "-m", "merge: 주문 요약", cwd=reviewer)
git("push", "origin", "main", cwd=reviewer)
print(git("log", "--oneline", "--graph", "-4", cwd=reviewer))
check = subprocess.run([sys.executable, "-c",
    "from summary import summarize; summarize('')"], cwd=reviewer, text=True, capture_output=True)
print("빈 문자열 종료 코드:", check.returncode)
print("입력 오류:", check.stderr.splitlines()[-1])
assert check.returncode != 0 and "TypeError: count는 정수여야 합니다" in check.stderr
assert git("rev-parse", "HEAD", cwd=reviewer) == git("rev-parse", "origin/main", cwd=reviewer)

*   3ad65c2 merge: 주문 요약
|\  
| * bd75d3d fix: 주문 건수 입력 확인
| * 84ce428 feat: 주문 요약 함수 추가
|/  
* 8eac6ee docs: 저장소와 PR 템플릿 시작
빈 문자열 종료 코드: 1
입력 오류: TypeError: count는 정수여야 합니다
